<div style="text-align: center; font-weight: bold;">
    <h1>EHR Preprocessing 4: Cohort Creation</h1>
    <h4>Author: Vidul Ayakulangara Panickan</h4>
</div>

**Goal:** By the end of this notebook, you will define a patient cohort based on a clinical condition (asthma), extract all codified and NLP features for that cohort from the cleaned and rolled-up data, and aggregate them into patient-level count matrices ready for downstream analysis.

**Recap:** Over the previous notebooks, we built a complete EHR preprocessing pipeline:

- **Getting Started** — downloaded MIMIC-IV data and explored its structure.
- **Data Cleaning** — standardized five raw datasets into a common four-column schema (`subject_id`, `date`, `code`, `coding_system`) and saved them as patient-level batch files.
- **Code Rollup** — mapped granular ICD, CPT, NDC, and ITEMID codes to clinically meaningful parent codes (PheCodes, CCS, RxNorm, LoincComponents).
- **NLP** — learned how to convert clinical free-text notes into structured UMLS CUI codes using dictionary-based NLP.

Now we bring it all together: we select a specific group of patients, pull their data, and reshape it into analysis-ready matrices.

> **Full executable notebook:** [github_repo/TUTORIAL/EHR_Preprocessing_4_Cohort_Creation.ipynb](github_repo/TUTORIAL/EHR_Preprocessing_4_Cohort_Creation.ipynb)

## 1. Setup

First, let's import the libraries we need and point to our workspace. Update `base_dir` to match your `EHR_TUTORIAL_WORKSPACE` location.

In [ ]:
import os
import pandas as pd

pd.set_option('display.max_columns', None)

# Update this path to your EHR_TUTORIAL_WORKSPACE location
base_dir = "EHR_TUTORIAL_WORKSPACE"

We need two output directories — one for codified features (PheCodes, CCS, RxNorm, LoincComponents) and one for NLP features (CUIs).

In [ ]:
cohort_dir = os.path.join(base_dir, 'processed_data', 'step5_cohort_aggregateddata')
codified_dir = os.path.join(cohort_dir, 'codified')
nlp_dir = os.path.join(cohort_dir, 'nlp')

os.makedirs(codified_dir, exist_ok=True)
os.makedirs(nlp_dir, exist_ok=True)

print(f"Codified output: {codified_dir}")
print(f"NLP output:      {nlp_dir}")

## 2. What Is Cohort Creation?

A **cohort** is a group of patients who share a common characteristic — typically a diagnosis, treatment, or exposure — that makes them the focus of a study. EHR-based studies are almost always conducted on a specific cohort rather than the full patient population. Common examples include:

- **Diagnosis-based cohorts:** patients diagnosed with a particular disease (e.g., asthma, diabetes)
- **Treatment-based cohorts:** patients who received a specific medication or procedure
- **Device-based cohorts:** patients implanted with a specific device

We do this because analyzing the full patient population would mix together unrelated conditions, making it hard to draw meaningful conclusions about any single one.

### What This Notebook Produces

Once we define our cohort, we extract and aggregate their data into two types of **patient-level count matrices**:

1. **Codified matrices** — one matrix per data type (Diagnoses, Procedures, Medications, Labs). Each row is a patient, each column is a code, and each cell counts how many times that code appeared for that patient.
2. **NLP matrix** — same structure, but columns are CUI codes extracted from clinical notes.

These matrices are the standard input format for downstream tasks like phenotyping, clustering, and predictive modeling.

## 3. Defining the Asthma Cohort

For this tutorial, we will build a cohort of patients with **asthma**.

A common strategy is to find patients using ICD codes for asthma. But ICD codes are very granular — there are dozens of ICD codes for different types and severities of asthma, and different studies often use different subsets, leading to inconsistencies.

A better approach is to use **PheCodes**. A PheCode (Phenotype Code) is a higher-level grouping that rolls up many related ICD codes into a single clinical concept. We already created these mappings in the Code Rollup notebook. Querying the [PheWAS Catalog](https://phewascatalog.org/) shows that **Asthma = PheCode 495**.

Our strategy: find all ICD codes that map to PheCode 495, then identify every patient who has at least one of those codes.

### Loading the Raw Diagnoses

We start from the raw `diagnoses_icd.csv` file (not the rolled-up version) because we need to match ICD codes against the PheCode mapping table. We also tag each row with its coding system (`ICD9` or `ICD10`) since the PheCode mapping requires this.

In [ ]:
hosp_dir = os.path.join(
    base_dir, "raw_data", "structured_data",
    "physionet.org", "files", "mimiciv", "3.1", "hosp"
)

diagnoses_icd = pd.read_csv(
    os.path.join(hosp_dir, "diagnoses_icd.csv"), dtype=str
).rename(columns={'icd_code': 'code'})

diagnoses_icd['coding_system'] = "ICD" + diagnoses_icd['icd_version']

print(f"Diagnoses: {diagnoses_icd.shape}")
display(diagnoses_icd.head())

Each row is one diagnosis event: a patient (`subject_id`) received a diagnosis code (`code`) during a hospital admission (`hadm_id`). The `coding_system` column tells us whether it is ICD-9 or ICD-10.

### Loading the PheCode Mapping

Next, we load the ICD-to-PheCode mapping file. This is the same mapping we used in the Code Rollup notebook.

In [ ]:
icd_to_phecode = pd.read_csv(
    os.path.join(base_dir, 'scripts', 'EHR-Processing-Tutorial-main',
                 'Rollup_Mappings', 'icd_to_phecode.csv'),
    dtype=str
)

print(f"PheCode mapping: {icd_to_phecode.shape}")
display(icd_to_phecode.head())

The mapping table has three columns: the raw ICD `code`, the `coding_system` (ICD9 or ICD10), and the corresponding `PheCode`. We will join on both `code` and `coding_system` to ensure ICD-9 codes map to ICD-9 PheCodes and ICD-10 codes map to ICD-10 PheCodes.

### Merging Diagnoses with PheCodes

In [ ]:
print(f"Diagnoses shape before merge: {diagnoses_icd.shape}")

comprehensive_diagnoses = pd.merge(
    diagnoses_icd, icd_to_phecode,
    on=['coding_system', 'code']
)

print(f"Diagnoses shape after merge:  {comprehensive_diagnoses.shape}")
display(comprehensive_diagnoses.head())

Notice the row count changed after the merge. This is an inner join — rows where the ICD code had no PheCode mapping were dropped, and rows where one ICD code maps to multiple PheCodes were expanded. This is expected behavior.

### Filtering to Asthma (PheCode 495)

In [ ]:
asthma_records = comprehensive_diagnoses[
    comprehensive_diagnoses['PheCode'] == '495'
]

print(f"Asthma diagnosis records: {asthma_records.shape}")
display(asthma_records.head())

We found over 42,000 asthma diagnosis records. But many patients have multiple asthma records (from different hospital visits), so the number of unique patients will be smaller.

### Extracting Unique Patients

In [ ]:
asthma_cohort = asthma_records[['subject_id']].drop_duplicates()

print(f"Unique patients in asthma cohort: {asthma_cohort.shape[0]}")
display(asthma_cohort.head())

We identified **20,316 unique patients** with at least one asthma-related ICD code (mapped to PheCode 495). This is our cohort.

> **Note on phenotyping specificity:** In practice, researchers often require at least **two** occurrences of a PheCode on separate dates to reduce false positives. For this tutorial, we use a single occurrence to keep the cohort large enough for demonstration.

## 4. Extracting Cohort Data: Diagnoses

Now that we know *who* is in our cohort, we need to extract *all* their clinical data — not just asthma codes, but every diagnosis, procedure, medication, and lab result.

We do this because downstream analyses (phenotyping, prediction, clustering) need the full clinical picture for each patient, not just the codes that defined the cohort.

We will walk through the full extraction and aggregation process on **Diagnoses** first, then wrap it into a reusable function for the other data types.

### Loading Rolled-Up Diagnoses

In the Code Rollup notebook, we saved rolled-up data as patient-level batch files. Each batch contains three columns: `subject_id`, `date`, and a parent code column (for diagnoses, this is `PheCode`). We load all batches and filter to our cohort patients.

> **Important:** The diagnoses data in MIMIC-IV does not include a specific timestamp — we inferred the timing as the admission date during Data Cleaning. Keep this in mind for downstream time-series analyses.

In [ ]:
rolledup_diagnoses_dir = os.path.join(
    base_dir, 'processed_data', 'step4_rolledup_finaldata', 'Diagnoses'
)
batch_files = os.listdir(rolledup_diagnoses_dir)

extracted_dfs = []
for batch_file in batch_files:
    batch = pd.read_csv(
        os.path.join(rolledup_diagnoses_dir, batch_file), dtype=str
    )
    extracted = pd.merge(batch, asthma_cohort, on='subject_id', how='inner')
    extracted_dfs.append(extracted)

extracted_diagnoses = pd.concat(extracted_dfs, ignore_index=True)

print(f"Extracted diagnoses for cohort: {extracted_diagnoses.shape}")
display(extracted_diagnoses.head())

We now have all diagnosis records (across all PheCodes, not just asthma) for the 20,316 patients in our cohort. Next, we aggregate these into a patient-level count matrix.

### Aggregating to Patient-Level Counts

We count how many times each PheCode appeared for each patient across all their visits. This transforms the data from a long format (one row per event) to a summary format.

In [ ]:
phecode_counts = extracted_diagnoses.groupby(
    ['subject_id', 'PheCode']
).size().reset_index(name='counts')

print(f"Patient-PheCode pairs: {phecode_counts.shape}")
display(phecode_counts.head(10))

Each row tells us how many times a specific PheCode appeared for a specific patient. For example, if patient `10001725` has PheCode `318` with a count of 2, that means this patient had two diagnosis events mapped to PheCode 318.

### Pivoting to a Count Matrix

Now we reshape from long format to a **wide matrix** where each row is a patient and each column is a PheCode. This is the standard format for most statistical and machine learning methods.

In [ ]:
diagnoses_matrix = phecode_counts.pivot_table(
    index='subject_id', columns='PheCode',
    values='counts', fill_value=0
)

print(f"Diagnoses matrix: {diagnoses_matrix.shape[0]} patients x {diagnoses_matrix.shape[1]} PheCodes")
display(diagnoses_matrix.head())

This tells us that across our 20,316 asthma patients, there are 1,725 unique PheCodes represented. Most cells are zero — a patient typically has only a small fraction of all possible diagnoses. This kind of **sparse matrix** is very common in EHR data.

### Saving the Diagnoses Matrix

In [ ]:
# Reset index so subject_id is saved as a column, not lost as the index
diagnoses_matrix.reset_index().to_csv(
    os.path.join(codified_dir, "Diagnoses.csv"), index=False
)

print(f"Saved to: {os.path.join(codified_dir, 'Diagnoses.csv')}")

## 5. Extracting the Remaining Data Types

The extraction process for Procedures, Medications, and Labs follows the exact same pattern we just walked through:

1. Load rolled-up batch files from the Code Rollup output
2. Filter to cohort patients via an inner merge
3. Group by patient and code, count occurrences
4. Pivot to a wide matrix
5. Save to CSV

Since the only difference between data types is the directory name and the code column name, we wrap these steps into a single reusable function.

In [ ]:
def extract_and_aggregate(data_type, code_col, cohort, base_dir, output_dir):
    """
    Extract cohort data from rolled-up batch files and aggregate
    into a patient-level count matrix.
    """
    rolledup_dir = os.path.join(
        base_dir, 'processed_data', 'step4_rolledup_finaldata', data_type
    )
    batch_files = os.listdir(rolledup_dir)

    extracted_dfs = []
    for batch_file in batch_files:
        batch = pd.read_csv(
            os.path.join(rolledup_dir, batch_file), dtype=str
        )
        extracted = pd.merge(batch, cohort, on='subject_id', how='inner')
        extracted_dfs.append(extracted)

    extracted = pd.concat(extracted_dfs, ignore_index=True)
    print(f"{data_type}: {extracted.shape[0]} records for {cohort.shape[0]} patients")

    counts = extracted.groupby(
        ['subject_id', code_col]
    ).size().reset_index(name='counts')

    matrix = counts.pivot_table(
        index='subject_id', columns=code_col,
        values='counts', fill_value=0
    )
    print(f"{data_type} matrix: {matrix.shape[0]} patients x {matrix.shape[1]} codes")

    matrix.reset_index().to_csv(
        os.path.join(output_dir, f"{data_type}.csv"), index=False
    )
    print(f"Saved to: {os.path.join(output_dir, data_type + '.csv')}")

    return matrix

This function does exactly what we did manually for Diagnoses. The `code_col` parameter tells it which column contains the parent code — `CCS` for procedures, `RxNorm` for medications, and `LoincComponent` for labs.

### Procedures

Procedure codes were rolled up to **CCS** (Clinical Classifications Software) categories — a grouping system from AHRQ that organizes thousands of procedure codes into ~230 clinically meaningful categories.

In [ ]:
procedures_matrix = extract_and_aggregate(
    data_type='Procedures',
    code_col='CCS',
    cohort=asthma_cohort,
    base_dir=base_dir,
    output_dir=codified_dir
)

display(procedures_matrix.head())

The procedures matrix is typically smaller (fewer unique codes) than diagnoses because CCS groups are broader.

### Medications

Medication codes were rolled up to **RxNorm** ingredient-level codes — standardized identifiers that group together different brand names, dosages, and formulations of the same active ingredient.

In [ ]:
medications_matrix = extract_and_aggregate(
    data_type='Medication',
    code_col='RxNorm',
    cohort=asthma_cohort,
    base_dir=base_dir,
    output_dir=codified_dir
)

display(medications_matrix.head())

Medications tend to have more unique codes than procedures because hospitals prescribe a wide variety of drugs. Some patients may have hundreds of medication records from multiple hospital stays.

### Labs

Lab codes were rolled up to **LOINC Components** — standardized names for what is being measured (e.g., "Glucose", "Hemoglobin") regardless of the specific test method.

In [ ]:
labs_matrix = extract_and_aggregate(
    data_type='Labs',
    code_col='LoincComponent',
    cohort=asthma_cohort,
    base_dir=base_dir,
    output_dir=codified_dir
)

display(labs_matrix.head())

Labs typically have the highest counts per patient because routine tests (blood counts, metabolic panels) are ordered frequently during hospital stays.

### Codified Data Summary

We have now extracted and aggregated all four codified data types for our asthma cohort:

| Data Type | Code System | Matrix Dimensions |
|-----------|-------------|-------------------|
| Diagnoses | PheCode | 20,316 patients x 1,725 codes |
| Procedures | CCS | (see output above) |
| Medications | RxNorm | (see output above) |
| Labs | LoincComponent | (see output above) |

Each matrix is saved in the `codified/` output directory. Next, we extract NLP features from clinical notes.

## 6. Extracting NLP Features from Clinical Notes

So far, all our features came from **structured** (codified) data — diagnosis codes, procedure codes, medication codes, and lab identifiers. But a large amount of clinical information is captured only in **free-text notes** written by doctors and nurses.

To use this information computationally, we need to convert text into structured codes. In Notebook 3 (NLP), we learned how to do this using dictionary-based NLP. Now we apply that same process to our asthma cohort's discharge summaries.

### MIMIC-IV Note vs. Hosp Version Mismatch

The clinical notes come from [MIMIC-IV Note v2.2](https://physionet.org/content/mimic-iv-note/2.2/), while the structured data comes from [MIMIC-IV Hosp v3.1](https://physionet.org/content/mimiciv/3.1/). Because these modules have different release versions, some patients in Hosp may not have notes, and vice versa.

> **Note:** This kind of mismatch is common in real-world health systems. A practical approach is to:
> 1. Restrict analyses to the intersection of patients with both note and structured data
> 2. Align timelines so that structured events do not extend beyond the last available note for each patient

### Key NLP Terms

Before we start, let's define three terms you will see throughout this section:

- **UMLS** (Unified Medical Language System): A large collection of biomedical vocabularies maintained by the National Library of Medicine. It provides a common framework for linking terms across different medical coding systems.
- **CUI** (Concept Unique Identifier): A code assigned by UMLS to each distinct medical concept. For example, "asthma" and "bronchial asthma" both map to the same CUI (`C0004096`) because they refer to the same concept.
- **ONCE** (Online Narrative and Codified feature Search Engine): A web app that generates disease-specific CUI dictionaries. Given a condition, it returns a list of related CUIs and their text synonyms.

### Installing petehr and Setting Up

We use [petehr](https://pypi.org/project/petehr/) — a Python toolkit built for this tutorial that performs dictionary-based NLP. It matches text against a CUI dictionary and returns the codes found in each note.

> **For production use**, we recommend [NILE](https://celehs.hms.harvard.edu/software/NILE.html) or [cTAKES](https://ctakes.apache.org/) for faster and more accurate NLP processing.

In [ ]:
!pip install petehr

from petehr import Text2Code

### Loading the CUI Dictionary

The CUI dictionary defines which medical concepts we want to detect in the notes. This asthma-specific dictionary was downloaded from the [ONCE app](https://shiny.parse-health.org/ONCE/) and contains CUIs related to asthma and its common comorbidities.

In [ ]:
asthma_dict_file = os.path.join(
    base_dir, 'scripts', 'EHR-Processing-Tutorial-main',
    'meta_files', 'Asthma_NLP_Dict.csv'
)

asthma_dictionary = pd.read_csv(asthma_dict_file, dtype=str)

print(f"Dictionary: {asthma_dictionary.shape}")
display(asthma_dictionary.head())

The dictionary has two columns: `STR` (the text string to match) and `CUI` (the UMLS concept it maps to). With ~2,980 term-CUI pairs covering 276 unique CUIs, the dictionary captures many synonyms — for example, "pulmonary hypertension", "hypertension pulmonary", and "pulmonary hypertension NOS" all map to the same CUI (`C0020542`).

### Initializing the Text-to-Code Converter

In [ ]:
text2cui = Text2Code(asthma_dict_file)

### Loading Discharge Notes

We load the discharge summaries from MIMIC-IV Note. These are the clinical notes written when a patient is discharged from the hospital, summarizing their entire stay.

In [ ]:
note_dir = os.path.join(
    base_dir, 'raw_data', 'note_data',
    'physionet.org', 'files', 'mimic-iv-note', '2.2', 'note'
)

discharge = pd.read_csv(
    os.path.join(note_dir, "discharge.csv"), dtype=str
)

print(f"Total discharge notes: {discharge.shape}")
display(discharge.head())

### Filtering to Cohort Patients

We filter the notes to keep only those belonging to our asthma cohort, and check how many cohort patients actually have note data.

In [ ]:
print(f"Cohort size: {len(asthma_cohort)} patients")

cohort_notes = pd.merge(
    asthma_cohort, discharge[['subject_id', 'charttime', 'text']],
    on='subject_id', how='inner'
)

patients_with_notes = cohort_notes['subject_id'].nunique()
coverage = patients_with_notes / len(asthma_cohort) * 100

print(f"Cohort notes: {cohort_notes.shape}")
print(f"Patients with notes: {patients_with_notes} ({coverage:.0f}% coverage)")

Out of 20,316 patients in our asthma cohort, approximately **73% have discharge notes**. The remaining 27% are missing notes due to the version mismatch between MIMIC-IV Note (v2.2) and Hosp (v3.1). Routine checks for data completeness like this are essential — they clarify the limitations of downstream NLP-based analyses.

### Cleaning the Note Data

We select only the columns we need, rename `charttime` to `date` for consistency with our codified data, and truncate timestamps to date-only format.

In [ ]:
print(f"Shape before dropping nulls: {cohort_notes.shape}")
cohort_notes = cohort_notes.dropna()
print(f"Shape after dropping nulls:  {cohort_notes.shape}")

cohort_notes = cohort_notes.rename(columns={'charttime': 'date'})
cohort_notes['date'] = cohort_notes['date'].str[:10]

display(cohort_notes[['subject_id', 'date']].head())

### Converting Notes to CUI Codes

Now we run the `Text2Code` converter on each note. For each note, it scans the text for terms in our dictionary and returns a comma-separated string of matching CUI codes. This step may take several minutes depending on the number of notes.

In [ ]:
cohort_notes['note_cui'] = cohort_notes['text'].map(
    lambda x: text2cui.convert(x)
)

display(cohort_notes[['subject_id', 'date', 'note_cui']].head())

Each row now has a `note_cui` column containing a comma-separated string of CUI codes found in that note. The same CUI can appear multiple times if the corresponding term was mentioned repeatedly.

### Reshaping CUI Strings into Individual Rows

To count CUIs per patient, we first need to split each comma-separated string into individual rows — one row per CUI per note. We then deduplicate so that each unique CUI appears only once per patient-date combination.

In [ ]:
# Split comma-separated CUI string into a list
cohort_notes['cui_list'] = cohort_notes['note_cui'].apply(
    lambda x: x.split(',') if x else None
)

# Keep only the columns we need going forward
cohort_notes = cohort_notes[['subject_id', 'date', 'cui_list']]

print(f"Shape before explode: {cohort_notes.shape}")

In [ ]:
# Expand so each CUI gets its own row
cohort_notes = cohort_notes.explode('cui_list')

print(f"Shape after explode: {cohort_notes.shape}")

In [ ]:
print(f"Shape before deduplication: {cohort_notes.shape}")

cohort_notes = cohort_notes.drop_duplicates()
cohort_notes = cohort_notes.rename(columns={'cui_list': 'cui'})

print(f"Shape after deduplication:  {cohort_notes.shape}")
display(cohort_notes.head())

The explode expanded ~50,000 note rows into ~42 million CUI rows (because each note contains many CUI mentions). After deduplication — removing repeated CUIs within the same patient-date — we are down to ~729,000 unique patient-date-CUI combinations.

### Aggregating CUIs into a Patient-Level Matrix

Just like we did for codified data, we count how many times each CUI appeared for each patient, then pivot into a wide matrix.

In [ ]:
cui_counts = cohort_notes.groupby(
    ['subject_id', 'cui']
).size().reset_index(name='counts')

print(f"Patient-CUI pairs: {cui_counts.shape}")
display(cui_counts.head())

In [ ]:
cui_matrix = cui_counts.pivot_table(
    index='subject_id', columns='cui',
    values='counts', fill_value=0
)

print(f"NLP matrix: {cui_matrix.shape[0]} patients x {cui_matrix.shape[1]} CUIs")
display(cui_matrix.head())

In [ ]:
# Reset index so subject_id is saved as a column
cui_matrix.reset_index().to_csv(
    os.path.join(nlp_dir, "CUI_counts.csv"), index=False
)

print(f"Saved to: {os.path.join(nlp_dir, 'CUI_counts.csv')}")

Notice the NLP matrix has fewer patients (~14,958) than the codified matrices (20,316) because only 73% of cohort patients have discharge notes. The matrix also has fewer columns (~229 CUIs) than the diagnoses matrix (~1,725 PheCodes) because our dictionary was specifically curated for asthma-related concepts.

## 7. Validation and Summary

Let's verify all output files were created and check their dimensions.

In [ ]:
print("=== Codified Matrices ===")
for f in sorted(os.listdir(codified_dir)):
    df = pd.read_csv(os.path.join(codified_dir, f), dtype=str, nrows=0)
    full_df = pd.read_csv(os.path.join(codified_dir, f), dtype=str)
    print(f"  {f}: {full_df.shape[0]} patients x {len(df.columns) - 1} codes")

print("\n=== NLP Matrix ===")
for f in sorted(os.listdir(nlp_dir)):
    full_df = pd.read_csv(os.path.join(nlp_dir, f), dtype=str)
    print(f"  {f}: {full_df.shape[0]} patients x {len(full_df.columns) - 1} CUIs")

The output directory structure looks like this:

```
EHR_TUTORIAL_WORKSPACE/
└── processed_data/
    └── step5_cohort_aggregateddata/
        ├── codified/
        │   ├── Diagnoses.csv
        │   ├── Procedures.csv
        │   ├── Medication.csv
        │   └── Labs.csv
        └── nlp/
            └── CUI_counts.csv
```

## What We Accomplished

In this notebook, we:

1. **Defined a patient cohort** — identified 20,316 patients with asthma (PheCode 495) from MIMIC-IV diagnoses data.
2. **Extracted codified features** — loaded rolled-up data from the Code Rollup notebook and filtered to our cohort across all four data types (Diagnoses, Procedures, Medications, Labs).
3. **Extracted NLP features** — ran dictionary-based NLP on discharge summaries to convert free text into structured CUI codes.
4. **Aggregated into count matrices** — transformed longitudinal event data into patient-level count matrices (patients × codes), the standard input for downstream analysis.
5. **Validated outputs** — confirmed all matrices were saved correctly with patient IDs preserved.

These matrices can now be used for phenotyping, predictive modeling, clustering, and other clinical research tasks.

## Additional Resources

1. [ONCE](https://shiny.parse-health.org/ONCE/) — A feature generation app that identifies related NLP and codified features based on an input disease. We used it to create the asthma CUI dictionary.
2. [KESER Network](https://shiny.parse-health.org/kesernetwork-linkage/) — A tool to identify and visualize codified concepts relevant to diseases, medications, and procedures.
3. [NILE](https://celehs.hms.harvard.edu/software/NILE.html) — A production-grade NLP tool for fast processing of clinical notes to UMLS CUIs. Requires a UMLS license.

---

> **Full executable notebook:** [github_repo/TUTORIAL/EHR_Preprocessing_4_Cohort_Creation.ipynb](github_repo/TUTORIAL/EHR_Preprocessing_4_Cohort_Creation.ipynb)